SHAP plots for SMOTE[nc] models

# Import libraries

In [1]:
import warnings
from numba.core.errors import NumbaDeprecationWarning
warnings.simplefilter(action='ignore', category=NumbaDeprecationWarning)

In [2]:
import shap
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from itertools import combinations

import sys
from pathlib import Path
sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from modelling3_utils import (
    MLPipeline, StatsModelsEstimator)

from tqdm import tqdm
from sklearn.inspection import permutation_importance
from pathlib import Path
from statsmodels.api import Logit
import matplotlib.cm as cm
from IPython.display import clear_output

In [3]:
RANDOM_STATE = 42

# SHAP plots

In [33]:
path_shap_plots = Path('../results/shap_comparison')
path_shap_plots_splashing = path_shap_plots / 'splashing_smote_selection'
os.makedirs(path_shap_plots_splashing, exist_ok=True)
path_shap_plots_no_fragmentation = path_shap_plots / 'no_fragmentation_smote_selection'
os.makedirs(path_shap_plots_no_fragmentation, exist_ok=True)

In [11]:
smote_postfixes = [
    '_opt-model',
    '_smote_opt-model',
    '_smote_opt-smote_opt-model',
    '_smotenc_opt-model',
    '_smotenc_opt-smote_opt-model']

In [6]:
df_metrics = pd.read_excel(r'..\results\metrics_modelling4.xlsx')

In [17]:
split_types = ['test', 'train']
metrics = []
for split_type in split_types:
    metrics.extend(
        [
            f'holdout_{split_type}_f1_macro',
            f'holdout_{split_type}_roc_auc',
            f'holdout_{split_type}_accuracy_balanced',
            f'cv_{split_type}_f1_macro_median',
            f'cv_{split_type}_roc_auc_median',
            f'cv_{split_type}_accuracy_balanced_median',
        ]
    )

## Splashing

In [22]:
target = 'splashing'
df_splashing = df_metrics[df_metrics['target']==target]
df_current_postfix = df_splashing[df_splashing['model'].apply(
    lambda x: x.split(target)[-1]==smote_postfixes[0])]
df_current_postfix = df_current_postfix.sort_values(
    by=metrics[:6], ascending=False)
df_current_postfix.head(3)

,dataset,target,model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_test_f1_macro,...,cv_train_recall_class_0_mean,cv_train_recall_class_0_median,cv_train_recall_class_0_std,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std
168,df_dimless,splashing,CatBoostClassifier_splashing_opt-model,"{'estimator': 'CatBoostClassifier', 'estimator...",0.866667,0.880000,0.916667,0.897959,0.950617,0.852826,...,1.00000,1.000000,0.000000,0.996529,0.995146,0.002195,"0.9999784157133607, 1.0, 0.9999785204914512, 0...",0.999985,0.999979,0.00001
176,df_dimless,splashing,LGBMClassifier_splashing_opt-model,"{'estimator': 'LGBMClassifier', 'estimator_par...",0.866667,0.865385,0.937500,0.900000,0.932485,0.850000,...,0.97853,0.973451,0.009299,0.948632,0.946602,0.011498,"0.9962659184113966, 0.995293110825845, 0.99551...",0.995815,0.995897,0.00052
88,df_dimless,splashing,KNeighborsClassifier_splashing_opt-model,"{'estimator': 'KNeighborsClassifier', 'estimat...",0.853333,0.862745,0.916667,0.888889,0.881173,0.836601,...,1.00000,1.000000,0.000000,0.996529,0.995146,0.002195,"0.9999784157133607, 1.0, 0.9999785204914512, 0...",0.999985,0.999979,0.00001


In [58]:
fnames_models = []
for postfix in smote_postfixes:
    df_current_postfix = df_splashing[df_splashing['model'].apply(
        lambda x: x.split(target)[-1]==postfix)]
    assert df_current_postfix.shape[0]>1
    df_current_postfix = df_current_postfix.sort_values(
        by=metrics[:6], ascending=False)
    fnames_models.append(df_current_postfix.iloc[0]["model"])
fnames_models

['CatBoostClassifier_splashing_opt-model',
 'CatBoostClassifier_splashing_smote_opt-model',
 'RandomForestClassifier_splashing_smote_opt-smote_opt-model',
 'LGBMClassifier_splashing_smotenc_opt-model',
 'AdaBoostClassifier_splashing_smotenc_opt-smote_opt-model']

In [ ]:
for fname in tqdm(fnames_models):
    index = 3
    if 'smote' in fname.lower():
        index = 4
    pipeline = joblib.load(f'../results/models_modelling4/{fname}')
    boost_model = pipeline.steps[index][1]
    ml_pipe = MLPipeline(
        target=target,
        estimator=boost_model,
        features_to_drop = (),
        model_postfix='test',)
    clear_output()
    X = ml_pipe.full_df.drop(columns=[target])
    X_transf = pipeline[:-2].transform(X)
    y = ml_pipe.full_df[target]
    if 'AdaBoost' in fname:
        explainer = shap.Explainer(boost_model.predict, X_transf)
    else:
        explainer = shap.Explainer(boost_model)
    shap_values = explainer(X_transf)
    if hasattr(X_transf, 'columns'):
        fnames = X_transf.columns
    elif hasattr(boost_model, 'feature_names_'): 
        fnames = boost_model.feature_names_
    elif hasattr(boost_model, 'feature_name_'): 
        fnames = boost_model.feature_name_
    elif hasattr(boost_model, 'feature_names_in_'):
        fnames = boost_model.feature_names_in_
    if len(shap_values.shape)==3:
        shap_values = shap_values[:, :, 0]
    shap.summary_plot(shap_values, X_transf, show=False, feature_names=fnames)
    short_name = fname.split('/')[-1]
    plt.title(short_name, fontsize=14)
    plt.savefig(path_shap_plots_splashing / f'{short_name}.jpg')
    plt.close()

ExactExplainer explainer: 373it [00:15,  8.52it/s]                         
100%|██████████| 5/5 [00:17<00:00,  3.44s/it]


## No fragmentation

In [61]:
target = 'no_fragmentation'
df_no_fragmentation = df_metrics[df_metrics['target']==target]
df_current_postfix = df_no_fragmentation[df_no_fragmentation['model'].apply(
    lambda x: x.split(target)[-1]==smote_postfixes[0])]
df_current_postfix = df_current_postfix.sort_values(
    by=metrics[:6], ascending=False)
df_current_postfix.head(3)

,dataset,target,model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_test_f1_macro,...,cv_train_recall_class_0_mean,cv_train_recall_class_0_median,cv_train_recall_class_0_std,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std
177,df_dimless,no_fragmentation,LGBMClassifier_no_fragmentation_opt-model,"{'estimator': 'LGBMClassifier', 'estimator_par...",0.920000,0.769231,1.0,0.869565,0.988182,0.905936,...,0.935287,0.935897,0.006632,0.989896,0.988235,0.004125,"0.9937423687423688, 0.9959276018099548, 0.9950...",0.995314,0.995123,0.001220
171,df_dimless,no_fragmentation,XGBClassifier_no_fragmentation_opt-model,"{'estimator': 'XGBClassifier', 'estimator_para...",0.906667,0.740741,1.0,0.851064,0.973636,0.891551,...,0.890110,0.888889,0.009614,0.969688,0.964706,0.008606,"0.9766737891737891, 0.980115635997989, 0.97473...",0.977186,0.976674,0.001895
175,df_dimless,no_fragmentation,RandomForestClassifier_no_fragmentation_opt-model,"{'estimator': 'RandomForestClassifier', 'estim...",0.906667,0.740741,1.0,0.851064,0.968182,0.891551,...,0.899267,0.901709,0.009379,0.979792,0.976471,0.005341,"0.987993487993488, 0.9887883358471594, 0.98386...",0.987166,0.987179,0.001503


In [62]:
fnames_models = []
for postfix in smote_postfixes:
    df_current_postfix = df_no_fragmentation[df_no_fragmentation['model'].apply(
        lambda x: x.split(target)[-1]==postfix)]
    assert df_current_postfix.shape[0]>1
    df_current_postfix = df_current_postfix.sort_values(
        by=metrics[:6], ascending=False)
    fnames_models.append(df_current_postfix.iloc[0]["model"])
fnames_models

['LGBMClassifier_no_fragmentation_opt-model',
 'RandomForestClassifier_no_fragmentation_smote_opt-model',
 'AdaBoostClassifier_no_fragmentation_smote_opt-smote_opt-model',
 'CatBoostClassifier_no_fragmentation_smotenc_opt-model',
 'RandomForestClassifier_no_fragmentation_smotenc_opt-smote_opt-model']

In [ ]:
for fname in tqdm(fnames_models):
    index = 3
    if 'smote' in fname.lower():
        index = 4
    pipeline = joblib.load(f'../results/models_modelling4/{fname}')
    boost_model = pipeline.steps[index][1]
    ml_pipe = MLPipeline(
        target=target,
        estimator=boost_model,
        features_to_drop = (),
        model_postfix='test',)
    clear_output()
    X = ml_pipe.full_df.drop(columns=[target])
    X_transf = pipeline[:-2].transform(X)
    y = ml_pipe.full_df[target]
    if 'AdaBoost' in fname:
        explainer = shap.Explainer(boost_model.predict, X_transf)
    else:
        explainer = shap.Explainer(boost_model)
    shap_values = explainer(X_transf)
    if hasattr(X_transf, 'columns'):
        fnames = X_transf.columns
    elif hasattr(boost_model, 'feature_names_'): 
        fnames = boost_model.feature_names_
    elif hasattr(boost_model, 'feature_name_'): 
        fnames = boost_model.feature_name_
    elif hasattr(boost_model, 'feature_names_in_'):
        fnames = boost_model.feature_names_in_
    if len(shap_values.shape)==3:
        shap_values = shap_values[:, :, 0]
    shap.summary_plot(shap_values, X_transf, show=False, feature_names=fnames)
    short_name = fname.split('/')[-1]
    plt.title(short_name, fontsize=14)
    plt.savefig(path_shap_plots_no_fragmentation / f'{short_name}.jpg')
    plt.close()